# User-friendly quickstart

The shortest recommended non-CP workflow: model → toy → fit → report → projections. This notebook is intentionally compact and is the suggested starting point for new analyses.


In [ ]:
import matplotlib.pyplot as plt
from dalitzplotfitter import (
    DecayChannel,DecayModel,FitSession,NonResonant,Parameter,RealImag,Resonance,
    enable_x64,generate_toy,plot_dalitz,plot_square_dalitz,
)
enable_x64()


In [ ]:
rho_x=Parameter.coefficient("rho.x",0.55,bounds=(-2,2),owner="rho",step=0.02)
rho_y=Parameter.coefficient("rho.y",0.10,bounds=(-2,2),owner="rho",step=0.02)
model=DecayModel(
    DecayChannel("B+",("K+","pi+","pi-")),
    [
        Resonance("Kstar",(0,2),RealImag(1,0),mass=0.8958,width=0.0474,spin=1),
        Resonance("rho",(1,2),RealImag(rho_x,rho_y),mass=0.7753,width=0.1491,spin=1),
        NonResonant(RealImag(-0.25,0.10)),
    ],
    normalization_method="square-dalitz",normalization_resolution=180,normalization_pair=(0,2),
)
truth={"rho.x":0.55,"rho.y":0.10}
data=generate_toy(model,20_000,parameters=truth,seed=1616,pool_size=150_000)


In [ ]:
plot_dalitz(data,x="s13",y="s23",title="Dalitz")
plt.show()
plot_square_dalitz(
    data,mother_mass=model.channel.parent_mass,masses=model.channel.daughter_masses,
    pair=(0,2),title="Square Dalitz"
)
plt.show()

session=FitSession(model,data)
result=session.fit({"rho.x":0.30,"rho.y":-0.05},simplex=True,ncall=30_000)
report=session.report(result)
session.plot_projection(result,"s13")
plt.show()
